# 🏥 NHS Claims Migration Validation
## Oracle → Spark SQL | Data Quality & Fidelity Checks

> **Note:** This is a portfolio demonstration project. All data is mocked and fictitious.  
> Inspired by real-world NHS-adjacent data migration work using Microsoft Fabric.

### What this notebook covers:
1. Load the mocked healthcare claims dataset
2. Simulate source (Oracle) and target (Spark) DataFrames
3. Run the full validation framework
4. Analyse results and visualise data quality issues
5. Demonstrate the Oracle → Spark SQL translation helper

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import sys
import os

# Add src directory to path
sys.path.insert(0, os.path.join('..', 'src'))

from migration_validator import MigrationValidator, ValidationReport
from sql_translation_helper import print_translation_reference, suggest_translations, scan_sql_file

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Setup complete ✅')

## 1. Load Mock Dataset

In [ ]:
# Load the mocked NHS claims dataset
claims = pd.read_csv('../data/healthcare_claims.csv', parse_dates=['service_date', 'submission_date'])

print(f'Loaded {len(claims):,} claims records')
print(f'Columns: {list(claims.columns)}')
print(f'Date range: {claims.service_date.min().date()} → {claims.service_date.max().date()}')
claims.head(3)

In [ ]:
# Quick data profile
print('--- NULL COUNTS ---')
print(claims.isnull().sum())

print('\n--- CLAIM STATUS DISTRIBUTION ---')
print(claims['claim_status'].value_counts())

## 2. Simulate Source vs Target DataFrames

In a real migration:
- **Source** = data extracted directly from Oracle via a JDBC connection or CSV export
- **Target** = data ingested into the Fabric Lakehouse and read back via Spark

Here we simulate both by using the same dataset, then intentionally introducing a small 
discrepancy in the 'target' to demonstrate how the validator catches issues.

In [ ]:
# SOURCE: represents Oracle extract (full, unmodified)
source_df = claims.copy()

# TARGET: represents Spark/Fabric ingested data
# We simulate a realistic scenario: a transformation accidentally dropped
# approved_amount for a small batch of records.
target_df = claims.copy()

# Simulate: 12 records lost their approved_amount during an ETL job
rng = np.random.default_rng(seed=99)
bad_idx = rng.choice(target_df[target_df['claim_status'] == 'APPROVED'].index, size=12, replace=False)
target_df.loc[bad_idx, 'approved_amount'] = np.nan

print(f'Source rows: {len(source_df):,}')
print(f'Target rows: {len(target_df):,}')
print(f'Simulated defect: {len(bad_idx)} approved_amount values nulled in target')

## 3. Run the Validation Framework

In [ ]:
# Initialise the validator
validator = MigrationValidator(
    source_df=source_df,
    target_df=target_df,
    table_name='nhs_claims',
    pk_column='claim_id',
    numeric_tolerance=0.001,  # 0.1% tolerance on aggregates
)

# Run all checks
report = validator.run_all_checks(
    numeric_columns=['claimed_amount', 'approved_amount', 'patient_age'],
    categorical_columns=['claim_status', 'patient_gender', 'nhs_trust_id'],
)

# Print the full report
MigrationValidator.print_report(report)

## 4. Export Results as a DataFrame

In [ ]:
results_df = MigrationValidator.to_dataframe(report)

# Summary table
print(results_df.groupby(['status', 'severity']).size().reset_index(name='count').to_string(index=False))

# Show failures and warnings only
print('\n--- ISSUES REQUIRING ATTENTION ---')
results_df[results_df['status'].isin(['FAIL', 'WARN'])][['check_name', 'status', 'severity', 'row_impact', 'detail']]

## 5. Visualise Data Quality

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('NHS Claims — Data Quality Overview', fontsize=14, fontweight='bold')

# --- Plot 1: Check results summary ---
status_counts = results_df['status'].value_counts()
colors = {'PASS': '#4CAF50', 'FAIL': '#F44336', 'WARN': '#FF9800', 'ERROR': '#9C27B0'}
bar_colors = [colors.get(s, 'grey') for s in status_counts.index]
axes[0, 0].bar(status_counts.index, status_counts.values, color=bar_colors, edgecolor='white')
axes[0, 0].set_title('Validation Check Results')
axes[0, 0].set_ylabel('Count')
for i, v in enumerate(status_counts.values):
    axes[0, 0].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# --- Plot 2: Claim status distribution ---
status_dist = claims['claim_status'].value_counts()
axes[0, 1].pie(status_dist.values, labels=status_dist.index, autopct='%1.1f%%',
               colors=['#4CAF50', '#2196F3', '#F44336', '#FF9800'])
axes[0, 1].set_title('Claim Status Distribution')

# --- Plot 3: Approved amount null analysis ---
approved_claims = claims[claims['claim_status'] == 'APPROVED']
null_count = approved_claims['approved_amount'].isna().sum()
not_null_count = approved_claims['approved_amount'].notna().sum()
axes[1, 0].bar(['Has Value', 'NULL'], [not_null_count, null_count], color=['#4CAF50', '#F44336'])
axes[1, 0].set_title('Approved Amount: APPROVED Claims')
axes[1, 0].set_ylabel('Record Count')
for i, v in enumerate([not_null_count, null_count]):
    axes[1, 0].text(i, v + 5, f'{v:,}', ha='center', fontweight='bold')

# --- Plot 4: Source vs target approved_amount sum by trust ---
src_by_trust = source_df.groupby('nhs_trust_id')['approved_amount'].sum().fillna(0)
tgt_by_trust = target_df.groupby('nhs_trust_id')['approved_amount'].sum().fillna(0)
x = range(len(src_by_trust))
width = 0.35
axes[1, 1].bar([i - width/2 for i in x], src_by_trust.values / 1e6, width, label='Source', color='#2196F3', alpha=0.8)
axes[1, 1].bar([i + width/2 for i in x], tgt_by_trust.values / 1e6, width, label='Target', color='#FF5722', alpha=0.8)
axes[1, 1].set_xticks(list(x))
axes[1, 1].set_xticklabels(src_by_trust.index)
axes[1, 1].set_title('Approved Amount by Trust (£M): Source vs Target')
axes[1, 1].set_ylabel('£ Millions')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('../reports/validation_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/validation_overview.png')

## 6. Oracle → Spark SQL Translation Helper

In [ ]:
# Show HIGH risk translations only
print_translation_reference(risk_filter='HIGH')

In [ ]:
# Scan Oracle SQL for migration issues
scan_sql_file('../sql/oracle/claims_analysis.sql')

In [ ]:
# Inline example: check an ad-hoc SQL snippet
sample_sql = """
SELECT
    NVL(approved_amount, 0) AS amount,
    NVL2(diagnosis_code, 'HAS_DIAG', 'NO_DIAG') AS flag,
    DECODE(claim_status, 'APPROVED', 1, 0) AS is_approved,
    TO_CHAR(service_date, 'YYYY-MM') AS month,
    TRUNC(SYSDATE) - TRUNC(submission_date) AS days_pending
FROM nhs_claims
"""

issues = suggest_translations(sample_sql)
print(f'Found {len(issues)} Oracle-specific patterns:\n')
for issue in issues:
    risk_icon = {'LOW': '🟢', 'MEDIUM': '🟡', 'HIGH': '🔴'}.get(issue['risk'], '⚪')
    print(f'{risk_icon} [{issue["risk"]}] {issue["detected"]}')
    print(f'   → {issue["suggestion"]}')
    print(f'   Note: {issue["notes"]}\n')

## 7. SQL Query Comparison

The cells below show the Oracle and Spark SQL equivalents side-by-side,
executed against our pandas DataFrames to demonstrate equivalent results.

In [ ]:
# Simulate Q1: Claims by trust — both Oracle and Spark logic produce same result in pandas

print('=== Q1: Total Approved Value by Trust ===')
print('Oracle SQL would use: NVL(approved_amount, 0)')
print('Spark SQL uses:       COALESCE(approved_amount, 0)')
print('Pandas equivalent:    .fillna(0)')
print()

approved = claims[claims['claim_status'] == 'APPROVED'].copy()
approved['approved_amount_filled'] = approved['approved_amount'].fillna(0)

result = (
    approved
    .groupby(['nhs_trust_id', 'nhs_trust_name'])
    .agg(
        total_claims=('claim_id', 'count'),
        total_approved_value=('approved_amount_filled', 'sum'),
        avg_approved_value=('approved_amount_filled', 'mean'),
    )
    .round(2)
    .sort_values('total_approved_value', ascending=False)
    .reset_index()
)

result['total_approved_value'] = result['total_approved_value'].map('£{:,.0f}'.format)
result['avg_approved_value'] = result['avg_approved_value'].map('£{:,.2f}'.format)
print(result.to_string(index=False))

In [ ]:
# Q4: Monthly volumes — Oracle TRUNC(date, 'MM') vs Spark DATE_TRUNC('month', date)

print('=== Q4: Monthly Claim Volumes ===')
print('Oracle: TRUNC(service_date, \'MM\')      → truncates to month start')
print('Spark:  DATE_TRUNC(\'month\', service_date) → same semantics')
print()

monthly = (
    claims
    .assign(service_month=claims['service_date'].dt.to_period('M'))
    .groupby('service_month')
    .agg(
        claim_count=('claim_id', 'count'),
        total_claimed=('claimed_amount', 'sum'),
    )
    .tail(12)  # show last 12 months
)

print(monthly.to_string())

## 8. Summary & Key Learnings

### What this framework validates:
- ✅ **Row counts** — detects data loss or duplication
- ✅ **Schema completeness** — catches missing or renamed columns  
- ✅ **Null analysis** — identifies unexpected NULL introduction in ETL
- ✅ **Aggregate checks** — confirms financial totals are preserved
- ✅ **Duplicate detection** — catches join fan-out issues
- ✅ **Category distributions** — validates reference data integrity
- ✅ **PK completeness** — verifies no records were lost/duplicated

### Key Oracle → Spark SQL differences encountered:
| Oracle | Spark | Risk |
|--------|-------|------|
| `NVL()` | `COALESCE()` | 🟡 Medium |
| `DECODE()` | `CASE WHEN` | 🔴 High |
| `TO_CHAR(date, 'YYYY-MM')` | `DATE_FORMAT(date, 'yyyy-MM')` | 🔴 High |
| `TRUNC(date, 'MM')` | `DATE_TRUNC('month', date)` | 🟡 Medium |
| `LISTAGG()` | `ARRAY_JOIN(COLLECT_SET())` | 🔴 High |
| `SYSDATE` | `current_date()` | 🟢 Low |